# R113 deep model and training-image audit

## tl;dr

- The main bottleneck is not model size or patch size.
- The boundary/necessity controller is dead, despite strong boundary information in the fixed input cues.
- Dense arrays dominate boundary-weighted training; more than half of flattened observations are clear/identity-like.
- Width and shift receive strongly conflicting gradients, while some mixed-hard labels are not identifiable from one image.
- Synthetic texture/noise does not match real 5x/20x/50x microscopy.
- The next authorized action is a small geometry/confidence-head microprobe, not a full model run.

## Context & Methods

The audit reads all 5,512 target/mask/SDF triplets, recreates six deterministic degradations per record, traces twelve checkpoints, measures loss gradients, and compares held-out microscopy texture.

### Key Assumptions

- Exact combined hashes control split leakage; normalized-shape hashes are only a collision screen.
- Threshold recoverability is favorable because it uses true polarity and midpoint.
- Gradient norms diagnose local optimization pressure, not exact AdamW attribution over all training steps.

In [1]:
from pathlib import Path
import json, pandas as pd
ROOT = Path(r'.')
def load(rel): return json.loads((ROOT / rel).read_text(encoding='utf-8'))
decision = load('results/FINAL_DEEP_AUDIT.json')
validation = load('results/final_validation.json')
validation

{'status': 'pass',
 'validated_local': '2026-08-30T14:03:07.230508+08:00',
 'checks': {'data_records_5512': True,
  'data_no_combined_cross_split_duplicates': True,
  'all_structural_masks_antialiased': True,
  'degradation_rows_33072': True,
  'training_contract_complete': True,
  'model_internal_checkpoints_12': True,
  'boundary_erf_zero_verified': True,
  'gradient_pooled_rows': True,
  'gradient_pairwise_rows': True,
  'representative_strata_rows': True,
  'clear_degraded_rows': True,
  'domain_gap_complete': True,
  'boundary_probe_complete': True,
  'all_source_hashes_unchanged': True},
 'failures': [],
 'scope': 'deep causal diagnosis and microprobe authorization; no full model promotion'}

## Data

Full-record quality and effective boundary weighting.

In [2]:
profile = load('results/data_profile/analysis_summary.json')
boundary_weight = pd.read_csv(ROOT / 'results/training_contract/family_effective_boundary_weight.csv')
boundary_weight.sort_values('edge_weight_share', ascending=False)[['family','records','record_share','edge_weight_share','perimeter_share']].head(8)

,family,records,record_share,edge_weight_share,perimeter_share
2,dense_array,379,0.0758,0.677494,0.753458
7,mixed_field,395,0.0790,0.053991,0.042521
10,ring,386,0.0772,0.041186,0.031533
6,lobed,407,0.0814,0.036067,0.027147
1,crescent,357,0.0714,0.026285,0.020461
9,rectangle,390,0.0780,0.024859,0.018586
0,circle,360,0.0720,0.024374,0.018234
5,irregular_blob,376,0.0752,0.023762,0.017745


In [3]:
recoverability = pd.DataFrame(load('results/degradation_recoverability/analysis_summary.json')['challenge_summary'])
recoverability[['challenge','records','source_threshold_iou_median','boundary_visibility_median','target_aligned_highfrequency_gain_median','unexplained_boundary_highfrequency_rms_median_gray']]

,challenge,records,source_threshold_iou_median,boundary_visibility_median,target_aligned_highfrequency_gain_median,unexplained_boundary_highfrequency_rms_median_gray
0,asymmetric_blur_halo,4457,0.951102,1.454963,0.113070,4.460288
1,boundary_fade,4457,0.602936,1.174797,0.131660,4.475333
2,identity,3090,0.994871,70.999923,1.000000,0.000000
3,low_contrast,4457,0.804292,1.281230,0.150835,4.574298
4,mixed_hard,4457,0.336781,0.867625,0.067738,4.477477
5,standard,4457,0.964165,1.917418,0.221737,4.542119
6,uneven_illumination,4457,0.746924,1.795346,0.223881,4.514152


## Results

Internal dynamics, gradient routing, boundary detectability, and real-domain distance.

In [4]:
internal = pd.read_csv(ROOT / 'results/model_internals/checkpoint_internal_dynamics.csv')
internal[['epoch','boundary_auc','boundary_spatial_std','learned_support_mean','learned_support_dominance_fraction','width_gain_mean','shift_abs_mean_px','renderer_tanh_saturation_fraction']]

,epoch,boundary_auc,boundary_spatial_std,learned_support_mean,learned_support_dominance_fraction,width_gain_mean,shift_abs_mean_px,renderer_tanh_saturation_fraction
0,4,0.494899,0.000024,0.432025,0.371433,1.044966,0.000000,0.000000
1,8,0.496234,0.000020,0.432049,0.371445,2.313227,0.547339,0.474677
2,12,0.496234,0.000020,0.432049,0.371445,2.495072,0.522146,0.478988
3,16,0.496234,0.000020,0.432049,0.371445,1.454203,0.492982,0.279651
4,20,0.496234,0.000020,0.432049,0.371445,1.551162,0.557783,0.491279
5,24,0.496234,0.000020,0.432049,0.371445,1.624652,0.587023,0.501264
6,28,0.496234,0.000020,0.432049,0.371445,1.633498,0.613938,0.567162
7,32,0.496234,0.000020,0.432049,0.371445,1.648528,0.615152,0.579316
8,36,0.496234,0.000020,0.432049,0.371445,1.647336,0.617280,0.578552
9,40,0.496234,0.000020,0.432049,0.371445,1.603881,0.622612,0.567964


In [5]:
gradients = pd.read_csv(ROOT / 'results/loss_gradients/pooled_loss_gradient_summary.csv')
gradients.sort_values(['parameter_group','effective_gradient_norm'], ascending=[True,False]).groupby('parameter_group').head(5)[['parameter_group','loss','effective_gradient_norm','alignment_with_expected_total']]

,parameter_group,loss,effective_gradient_norm,alignment_with_expected_total
13,boundary_head,contour,4.134667e-03,1.000000
17,boundary_head,direct,2.201463e-03,1.000000
19,boundary_head,gradient,1.215359e-03,1.000000
18,boundary_head,reconstruction,1.007884e-03,1.000000
14,boundary_head,contour_bce,3.093335e-04,1.000000
3,detector_backbone,centroid,1.975246e-08,0.875715
0,detector_backbone,contour,1.320734e-08,0.950330
6,detector_backbone,gradient,8.374510e-09,-0.717540
2,detector_backbone,area,4.991103e-09,0.945452
4,detector_backbone,direct,3.120943e-09,-0.097042


In [6]:
conflict = pd.read_csv(ROOT / 'results/loss_gradients/clear_degraded_gradient_conflict.csv')
conflict[conflict.loss == 'expected_sum_of_six_base_losses'][['parameter_group','clear_over_degraded_norm','clear_vs_degraded_cosine']]

,parameter_group,clear_over_degraded_norm,clear_vs_degraded_cosine
6,renderer_backbone,0.014704,-0.004536
13,width_head,0.001075,-0.463291
20,shift_head,5.917006,-0.160725


In [7]:
detect = load('results/boundary_detectability/analysis_summary.json')
{'current_R109_boundary_AUC': detect['current_r109_boundary_auc_reference'], 'fixed_cue_logistic_test_AUC': detect['overall']['test_auc'], 'dense_array_median_AUC': next(row['auc_median'] for row in detect['family_results'] if row['family']=='dense_array')}

{'current_R109_boundary_AUC': 0.4962337939465492,
 'fixed_cue_logistic_test_AUC': 0.8698795744015628,
 'dense_array_median_AUC': 0.419921875}

In [8]:
domain = load('results/domain_gap/analysis_summary.json')['distance_analysis']['real_domains']
pd.DataFrame(domain).T[['patches','distance_over_synthetic_self_median']]

,patches,distance_over_synthetic_self_median
junhua_ring,64,2.464908
real_20x,9,3.754582
real_50x,10,3.631014
real_5x,9,8.811436
sem_optical,300,1.348217


## Takeaways

1. Rebalance by per-scene boundary mass and isolate dense-array quotas.
2. Remove automatic clear pairing from every record; schedule smaller dedicated identity batches.
3. Replace constant-prone boundary BCE/Dice with joint SDF, normal, confidence, and recoverability supervision.
4. Keep the controller trainable with the renderer and remove mandatory raw-gradient support floors.
5. Train width/shift from spatial geometry targets, then train bounded detail separately only where information is recoverable.
6. Use correlated multiscale microscopy-like texture/noise; do not use real SEM as labels.
7. Run the small controller microprobe before any full new model.